# Week 1: PDF text extraction and preprocessing
Run the cells in order. Set `PDF_PATH` to a research paper PDF on your computer.

In [ ]:
from pathlib import Path
import re
import fitz  # PyMuPDF

PDF_PATH = Path('sample_paper.pdf')  # Change this to your PDF file path


## 1. Extract text from the PDF
This extracts selectable text. Scanned PDFs may need OCR, which is outside this first notebook.

In [ ]:
def extract_pdf_text(pdf_path: Path) -> str:
    if not pdf_path.exists():
        raise FileNotFoundError(f'PDF not found: {pdf_path.resolve()}')
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_number, page in enumerate(doc, start=1):
            text = page.get_text('text')
            pages.append(f'\\n\\n--- Page {page_number} ---\\n{text}')
    return ''.join(pages)

raw_text = extract_pdf_text(PDF_PATH)
print(raw_text[:3000])

## 2. Clean basic formatting noise
Keep the raw text as well. Cleaning can accidentally remove meaningful symbols, citations, or equation formatting.

In [ ]:
def clean_text(text: str) -> str:
    text = text.replace('\\x00', ' ')
    text = re.sub(r'[ \\t]+', ' ', text)
    text = re.sub(r'\\n{3,}', '\\n\\n', text)
    return text.strip()

cleaned_text = clean_text(raw_text)
print(cleaned_text[:3000])

## 3. Split text into overlapping chunks
Overlap helps preserve context when an idea crosses a chunk boundary. This simple character-based approach is for learning; later we can compare token-based chunking.

In [ ]:
def make_chunks(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    if chunk_size <= 0:
        raise ValueError('chunk_size must be positive')
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError('overlap must be between 0 and chunk_size - 1')
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = end - overlap
    return chunks

chunks = make_chunks(cleaned_text)
print('Number of chunks:', len(chunks))
print('First chunk preview:\\n', chunks[0][:800] if chunks else 'No text extracted')

## 4. Save chunks for later weeks

In [ ]:
import json

output_path = Path('paper_chunks.json')
output_path.write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {len(chunks)} chunks to {output_path.resolve()}')

## Week 1 tasks
1. Try the notebook with two different research papers.
2. Compare the extracted text with the PDF pages.
3. Record at least two extraction issues.
4. Change `chunk_size` and `overlap`; observe how the number and content of chunks change.

**Reminder:** Text extraction is imperfect. Always inspect the extracted text before using it in a search or question-answering system.